In [1]:
import geemap
import ee
geemap.ee_initialize()
ee.Authenticate()

True

In [5]:
import ee
import geemap

# -----------------------------
# 1) Initialize Earth Engine
# -----------------------------
try:
    ee.Initialize()
except Exception:
    geemap.ee_initialize()
    ee.Initialize()

# -----------------------------
# 2) ROI (Haor bbox)
# -----------------------------
BBOX_BD = [89.8, 23.5, 92.8, 25.5]  # [west, south, east, north]
roi = ee.Geometry.Rectangle(BBOX_BD)

# -----------------------------
# 3) Dates (edit)
# -----------------------------
REF_START    = "2025-04-01"
REF_END      = "2025-04-30"
LATEST_START = "2025-05-20"
LATEST_END   = "2025-06-30"

# If too few images, increase to 100
MAX_CLOUD_PCT = 90

# Visualization
VIS_TRUECOLOR = {"bands": ["B4", "B3", "B2"], "min": 0, "max": 3000, "gamma": 1.2}

# -----------------------------
# 4) Cloud/shadow mask (Sentinel-2 SR Harmonized)
# -----------------------------
def mask_s2_sr(img):
    qa = img.select("QA60")
    cloud_bit  = 1 << 10
    cirrus_bit = 1 << 11
    mask_qa = qa.bitwiseAnd(cloud_bit).eq(0).And(qa.bitwiseAnd(cirrus_bit).eq(0))

    scl = img.select("SCL")
    mask_scl = (
        scl.neq(3)   # shadow
        .And(scl.neq(8))    # medium cloud
        .And(scl.neq(9))    # high cloud
        .And(scl.neq(10))   # cirrus
        .And(scl.neq(11))   # snow/ice
    )

    return img.updateMask(mask_qa).updateMask(mask_scl)

def add_ndvi(img):
    ndvi = img.normalizedDifference(["B8", "B4"]).rename("NDVI")
    return img.addBands(ndvi)

# -----------------------------
# 5) Build a "stacked" composite for a window
# -----------------------------
def stacked_truecolor_composite(roi, start, end, max_cloud_pct=90, method="median"):
    """
    method:
      - "median"  : robust true color composite (recommended)
      - "mosaic"  : fast, last-on-top mosaic (can be patchy)
      - "quality" : quality mosaic using NDVI (often sharp but can bias water)
    """
    col = (
        ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
        .filterBounds(roi)
        .filterDate(start, end)
        .filter(ee.Filter.lte("CLOUDY_PIXEL_PERCENTAGE", max_cloud_pct))
        .map(mask_s2_sr)
        .map(lambda img: img.clip(roi))
    )

    n = col.size()

    # fallback empty RGB
    empty = ee.Image.constant([0, 0, 0]).rename(["B4", "B3", "B2"]).updateMask(ee.Image.constant(0)).clip(roi)

    def build():
        if method == "mosaic":
            comp = col.select(["B4", "B3", "B2"]).mosaic()
        elif method == "quality":
            comp = col.map(add_ndvi).qualityMosaic("NDVI").select(["B4", "B3", "B2"])
        else:
            # "median" default
            comp = col.select(["B4", "B3", "B2"]).median()

        # add a quick "valid pixel" mask band for diagnostics
        valid = comp.select("B4").mask().rename("valid")
        return ee.Dictionary({"image": comp, "count": n, "valid": valid})

    return ee.Dictionary(ee.Algorithms.If(n.gt(0), build(), ee.Dictionary({"image": empty, "count": n, "valid": empty.select("B4").mask().rename("valid")})))

# -----------------------------
# 6) Build REF and LATEST composites
# -----------------------------
METHOD = "median"  # try: "median" (best), "mosaic" (fast), "quality" (sharp)

ref_dict    = stacked_truecolor_composite(roi, REF_START, REF_END, MAX_CLOUD_PCT, method=METHOD)
latest_dict = stacked_truecolor_composite(roi, LATEST_START, LATEST_END, MAX_CLOUD_PCT, method=METHOD)

ref_img    = ee.Image(ref_dict.get("image"))
latest_img = ee.Image(latest_dict.get("image"))

ref_n    = ee.Number(ref_dict.get("count")).getInfo()
latest_n = ee.Number(latest_dict.get("count")).getInfo()
print(f"[DIAG] REF window images after filters:    {ref_n}")
print(f"[DIAG] LATEST window images after filters: {latest_n}")
print(f"[DIAG] Composite method: {METHOD} | MAX_CLOUD_PCT={MAX_CLOUD_PCT}")

# Optional: estimate fraction of ROI with valid pixels (after masking + compositing)
def valid_fraction(valid_mask_img):
    # mean of 0/1 mask over ROI at coarse scale (avoid heavy compute)
    frac = valid_mask_img.unmask(0).reduceRegion(
        reducer=ee.Reducer.mean(),
        geometry=roi,
        scale=500,
        maxPixels=1e7
    ).get("valid")
    return ee.Number(frac)

ref_valid = valid_fraction(ee.Image(ref_dict.get("valid"))).getInfo()
lat_valid = valid_fraction(ee.Image(latest_dict.get("valid"))).getInfo()
print(f"[DIAG] REF valid-pixel fraction (rough):    {ref_valid:.3f}")
print(f"[DIAG] LATEST valid-pixel fraction (rough): {lat_valid:.3f}")

# -----------------------------
# 7) Visualize
# -----------------------------
m = geemap.Map()
m.centerObject(roi, 7)

m.addLayer(ref_img,    VIS_TRUECOLOR, f"S2 True Color REF ({REF_START} to {REF_END})", True)
m.addLayer(latest_img, VIS_TRUECOLOR, f"S2 True Color LATEST ({LATEST_START} to {LATEST_END})", True)
m.addLayer(ee.Image().paint(roi, 1, 2), {"palette": ["FFFFFF"]}, "ROI", True)

m

[DIAG] REF window images after filters:    194
[DIAG] LATEST window images after filters: 145
[DIAG] Composite method: median | MAX_CLOUD_PCT=90
[DIAG] REF valid-pixel fraction (rough):    0.984
[DIAG] LATEST valid-pixel fraction (rough): 0.805


Map(center=[24.504754997740452, 91.30000000000003], controls=(WidgetControl(options=['position', 'transparent_…